In [15]:
import polars as pl
import datetime as dt
import os
from pathlib import Path

In [16]:
L2DATA_PATH = "/data/xujiayi/xjy/l2/test"

In [17]:
def normalize_date(date: dt.date | dt.datetime | str) -> str:
    if isinstance(date, (dt.datetime, dt.date)):
        return date.strftime("%Y%m%d")
    return str(date).replace("-", "").replace(".", "").strip("/")

In [18]:
date = normalize_date('20250930')

filepath = Path(L2DATA_PATH)/"raw"/date

outpath = Path(L2DATA_PATH)/"proc"/date
outpath.mkdir(parents=True, exist_ok=True)

#### 深交所三表整理

In [9]:
szcj_merged = pl.read_parquet(filepath/'szcj.pq').filter(pl.col('MDStreamID')==11).drop(['MDStreamID','SecurityIDSource','LocalTime','SeqNo']).rename({
    'LastPx':'Price',
    'LastQty':'OrderQty'
}).with_columns([
    pl.col('TransactTime').str.to_time(format="%H:%M:%S%.3f"),
    pl.when(pl.col('BidApplSeqNum')>pl.col('OfferApplSeqNum')).then(pl.lit(1)).otherwise(pl.lit(-1)).alias('Side')
])
szcj = szcj_merged.filter(pl.col('ExecType')==70).drop('ExecType')
szcancel = szcj_merged.filter(pl.col('ExecType')==52).drop('ExecType')

In [10]:
szcj.write_parquet(outpath/'szcj.pq',compression="gzip")
szcancel.write_parquet(outpath/'szcancel.pq',compression="gzip")

In [11]:
szwt = pl.read_parquet(filepath/'szwt.pq').filter(pl.col('MDStreamID')==11).drop(['MDStreamID','SecurityIDSource','LocalTime','SeqNo','OrdType'])
szwt = szwt.with_columns([
    pl.col('Side').replace(49,1).replace(50,-1).cast(pl.Int8),
    pl.col('TransactTime').str.to_time(format="%H:%M:%S%.3f")
])

In [12]:
szwt.write_parquet(outpath/'szwt.pq',compression="gzip")

#### 上交所三表整理

In [31]:
sh = pl.read_parquet(filepath/'sh.pq').drop(['TradeMoney','LocalTime','SeqNo','TickBSFlag']).rename({
    'BizIndex':'ApplSeqNum',
    'Channel':'ChannelNo',
    'TickTime':'TransactTime',
    'Qty':'OrderQty'
}).with_columns([
    pl.col('TransactTime').str.to_time(format="%H:%M:%S%.3f"),
])
sh

ApplSeqNum,ChannelNo,SecurityID,TransactTime,Type,BuyOrderNO,SellOrderNO,Price,OrderQty
i64,i64,i64,time,str,i64,i64,f64,i64
1,1,751028,06:00:00.110,"""S""",0,0,0.0,0
2,1,751900,06:00:00.110,"""S""",0,0,0.0,0
3,1,751004,06:00:00.110,"""S""",0,0,0.0,0
4,1,751019,06:00:00.110,"""S""",0,0,0.0,0
5,1,751992,06:00:00.110,"""S""",0,0,0.0,0
…,…,…,…,…,…,…,…,…
28262536,6,560820,15:05:00.560,"""S""",0,0,0.0,0
28262537,6,588130,15:05:00.560,"""S""",0,0,0.0,0
28262538,6,603210,15:05:00.560,"""S""",0,0,0.0,0


In [32]:
shwt_added = sh.filter(pl.col('Type')=='A').drop('Type').with_columns([
    pl.when(pl.col('BuyOrderNO')!=0).then(pl.col('BuyOrderNO')).otherwise(pl.col('SellOrderNO')).alias('OrderNo'),
    pl.when(pl.col('BuyOrderNO')!=0).then(pl.lit(1)).otherwise(pl.lit(-1)).alias('Side'),
]).drop(['BuyOrderNO','SellOrderNO'])
shwt_added 

ApplSeqNum,ChannelNo,SecurityID,TransactTime,Price,OrderQty,OrderNo,Side
i64,i64,i64,time,f64,i64,i64,i32
660,2,688545,09:15:01.970,36.01,300,12330,1
661,2,688545,09:15:02.550,41.42,1704,20176,-1
662,2,688545,09:15:02.550,45.23,9,21026,-1
663,2,688545,09:15:02.550,37.02,1000,21338,1
664,2,688545,09:15:02.680,36.71,1000,21373,1
…,…,…,…,…,…,…,…
31571741,5,603202,14:59:42.510,98.73,100,19318579,-1
31571742,5,603202,14:59:45,96.87,100,19320909,1
31571743,5,603202,14:59:52.550,96.77,200,19329054,-1


In [33]:
shcj = sh.filter(
    (pl.col('Type')=='T') | (pl.col('Type')=='D')
).join(
    shwt_added.select(['ChannelNo','OrderNo','ApplSeqNum','SecurityID']),  # 获取买单的频道内索引
    left_on=['ChannelNo','BuyOrderNO','SecurityID'],
    right_on=['ChannelNo','OrderNo','SecurityID'],
    how='left',
    suffix='_buy'
).join(
    shwt_added.select(['ChannelNo','OrderNo','ApplSeqNum','SecurityID']),  # 获取卖单的频道内索引
    left_on=['ChannelNo','SellOrderNO','SecurityID'],
    right_on=['ChannelNo','OrderNo','SecurityID'],
    how='left',
    suffix='_sell'
).rename({
    'ApplSeqNum_buy':'BidApplSeqNum',
    'ApplSeqNum_sell':'OfferApplSeqNum'
}).with_columns([
    pl.when(pl.col('Type')=='D').then(pl.col('BidApplSeqNum').fill_null(0)).otherwise(pl.col('BidApplSeqNum').fill_null(pl.col('ApplSeqNum'))),
    pl.when(pl.col('Type')=='D').then(pl.col('OfferApplSeqNum').fill_null(0)).otherwise(pl.col('OfferApplSeqNum').fill_null(pl.col('ApplSeqNum'))),
]).with_columns([                                               # 判断买卖方向
    pl.when(pl.col('BidApplSeqNum')>pl.col('OfferApplSeqNum')).then(pl.lit(1)).otherwise(pl.lit(-1)).alias('Side')
]).drop(['BuyOrderNO','SellOrderNO'])
shcj

ApplSeqNum,ChannelNo,SecurityID,TransactTime,Type,Price,OrderQty,BidApplSeqNum,OfferApplSeqNum,Side
i64,i64,i64,time,str,f64,i64,i64,i64,i32
810,2,688545,09:15:11.450,"""D""",38.17,12717,0,699,-1
860,2,688545,09:15:17.280,"""D""",37.49,267,0,859,-1
870,2,688545,09:15:18.380,"""D""",45.23,350,0,868,-1
899,2,688545,09:15:28.010,"""D""",37.7,1325,894,0,1
901,2,688545,09:15:29.850,"""D""",38.0,778,0,900,-1
…,…,…,…,…,…,…,…,…,…
31571810,5,603202,15:00:01.690,"""T""",98.75,400,31571725,31571710,1
31571811,5,603202,15:00:01.690,"""T""",98.75,100,31571733,31571710,1
31571812,5,603202,15:00:01.690,"""T""",98.75,1000,31571735,31571710,1


In [34]:
shcancel = shcj.filter(pl.col('Type')=='D').drop('Type')
shcancel

ApplSeqNum,ChannelNo,SecurityID,TransactTime,Price,OrderQty,BidApplSeqNum,OfferApplSeqNum,Side
i64,i64,i64,time,f64,i64,i64,i64,i32
810,2,688545,09:15:11.450,38.17,12717,0,699,-1
860,2,688545,09:15:17.280,37.49,267,0,859,-1
870,2,688545,09:15:18.380,45.23,350,0,868,-1
899,2,688545,09:15:28.010,37.7,1325,894,0,1
901,2,688545,09:15:29.850,38.0,778,0,900,-1
…,…,…,…,…,…,…,…,…
28086978,6,516510,14:59:59.930,1.673,280300,28085229,0,1
28086979,6,516510,14:59:59.930,1.67,280300,28048955,0,1
28086981,6,513700,14:59:59.950,0.77,3000,26342070,0,1


In [35]:
shcancel.write_parquet(outpath/'shcancel.pq',compression="gzip")

In [36]:
shcj = shcj.filter(pl.col('Type')=='T').drop('Type')
shcj

ApplSeqNum,ChannelNo,SecurityID,TransactTime,Price,OrderQty,BidApplSeqNum,OfferApplSeqNum,Side
i64,i64,i64,time,f64,i64,i64,i64,i32
1210,2,688545,09:25:00.010,38.75,50,1021,869,1
1211,2,688545,09:25:00.010,38.75,393,1021,953,1
1212,2,688545,09:25:00.010,38.75,200,1021,983,1
1213,2,688545,09:25:00.010,38.75,157,1021,985,1
1214,2,688545,09:25:00.010,38.75,1000,1052,985,1
…,…,…,…,…,…,…,…,…
31571810,5,603202,15:00:01.690,98.75,400,31571725,31571710,1
31571811,5,603202,15:00:01.690,98.75,100,31571733,31571710,1
31571812,5,603202,15:00:01.690,98.75,1000,31571735,31571710,1


In [37]:
shcj.write_parquet(outpath/'shcj.pq',compression="gzip")

In [38]:
shwt_added = shwt_added.drop('OrderNo')
shwt_added

ApplSeqNum,ChannelNo,SecurityID,TransactTime,Price,OrderQty,Side
i64,i64,i64,time,f64,i64,i32
660,2,688545,09:15:01.970,36.01,300,1
661,2,688545,09:15:02.550,41.42,1704,-1
662,2,688545,09:15:02.550,45.23,9,-1
663,2,688545,09:15:02.550,37.02,1000,1
664,2,688545,09:15:02.680,36.71,1000,1
…,…,…,…,…,…,…
31571741,5,603202,14:59:42.510,98.73,100,-1
31571742,5,603202,14:59:45,96.87,100,1
31571743,5,603202,14:59:52.550,96.77,200,-1


In [ ]:
def RestoreOrder(wt, cj):
    
    cj = cj.sort(["ChannelNo", "ApplSeqNum", "SecurityID", "TransactTime"])

    # 剔除集合竞价
    cj_df = cj.filter(~((pl.col("TransactTime") <= pl.time(9, 25)) | (pl.col("TransactTime") >= pl.time(14, 57))))

    # 1. 从成交表提取买卖双方订单并汇总
    cj_buy = (
        cj_df
        .select([
            "ChannelNo",
            pl.col("BidApplSeqNum").alias("ApplSeqNum"),
            "SecurityID",
            pl.when(pl.col("BidApplSeqNum") > pl.col("OfferApplSeqNum"))
            .then(pl.col("OrderQty"))
            .otherwise(0)
            .alias("OrderQty"),
            "Price",
            "TransactTime",
        ])
        .with_columns(pl.lit(1).alias("Side"))
    )
    cj_sell = (
        cj_df
        .select([
            "ChannelNo",
            pl.col("OfferApplSeqNum").alias("ApplSeqNum"),
            "SecurityID",
            pl.when(pl.col("OfferApplSeqNum") > pl.col("BidApplSeqNum"))
            .then(pl.col("OrderQty"))
            .otherwise(0)
            .alias("OrderQty"),
            "Price",
            "TransactTime",
        ])
        .with_columns(pl.lit(-1).alias("Side"))
    )
    cj_all = pl.concat([cj_buy, cj_sell]).filter(pl.col("OrderQty") > 0)
    cj_summary = cj_all.group_by(["ChannelNo", "ApplSeqNum", "SecurityID", "Side"]).agg([
        pl.sum("OrderQty"),
        pl.last("Price"),  # 一笔主动单可能同时吃掉多笔挂单
        pl.last("TransactTime")
    ])

    # 2. 使用反连接和半连接分离三种情况
    # 部分成交：同时存在于委托和成交（inner join）
    partial = wt.join(
        cj_summary.select(["ChannelNo", "ApplSeqNum", "SecurityID", "Side", "OrderQty"]),
        on=["ChannelNo", "ApplSeqNum", "SecurityID", "Side"],
        how="inner"
    ).with_columns([
        (pl.col("OrderQty") + pl.col("OrderQty_right")).alias("OrderQty"),
        pl.lit("主动部分成交").alias("OrderStatus")
    ]).drop("OrderQty_right")

    # 未成交：存在于委托但不存在于成交（anti join）
    untouched = wt.join(
        cj_summary.select(["ChannelNo", "ApplSeqNum", "SecurityID", "Side"]),
        on=["ChannelNo", "ApplSeqNum", "SecurityID", "Side"],
        how="anti"
    ).with_columns(
        pl.lit("挂单被动成交").alias("OrderStatus")
    )
    untouched = untouched.select(partial.columns)

    # 完全成交：存在于成交但不存在于委托（anti join）
    new = cj_summary.join(
        wt.select(["ChannelNo", "ApplSeqNum", "SecurityID", "Side"]),
        on=["ChannelNo", "ApplSeqNum", "SecurityID", "Side"],
        how="anti"
    ).with_columns([
        pl.lit("主动完全成交").alias("OrderStatus"),
    ])
    new = new.select(partial.columns)

    # 3. 合并所有订单
    init_order = pl.concat([partial, untouched, new])
    return init_order

In [50]:
def RestoreOrder(wt, cj):
    cj = cj.sort(["ChannelNo", "ApplSeqNum", "SecurityID", "TransactTime"])

    # 剔除集合竞价。集合竞价阶段 A 行 Qty 按说明已经是原始委托量，不需要用成交加回。
    cj_df = cj.filter(
        ~(
            (pl.col("TransactTime") <= pl.time(9, 25))
            | (pl.col("TransactTime") >= pl.time(14, 57))
        )
    )

    # 1. 只提取主动方成交用于还原原始委托量
    # BidApplSeqNum > OfferApplSeqNum: 买方主动
    # OfferApplSeqNum > BidApplSeqNum: 卖方主动
    active_buy = (
        cj_df
        .filter(pl.col("BidApplSeqNum") > pl.col("OfferApplSeqNum"))
        .select([
            "ChannelNo",
            pl.col("BidApplSeqNum").alias("ApplSeqNum"),
            "SecurityID",
            "OrderQty",
            "Price",
            "TransactTime",
            pl.col("ApplSeqNum").alias("TradeApplSeqNum"),
        ])
        .with_columns(pl.lit(1).alias("Side"))
    )

    active_sell = (
        cj_df
        .filter(pl.col("OfferApplSeqNum") > pl.col("BidApplSeqNum"))
        .select([
            "ChannelNo",
            pl.col("OfferApplSeqNum").alias("ApplSeqNum"),
            "SecurityID",
            "OrderQty",
            "Price",
            "TransactTime",
            pl.col("ApplSeqNum").alias("TradeApplSeqNum"),
        ])
        .with_columns(pl.lit(-1).alias("Side"))
    )

    active_cj = pl.concat([active_buy, active_sell])

    # 2. 对存在 A 行的订单，只加回 A 行之前的主动成交。
    # 连续竞价中若先成交再剩余挂单，交易所顺序是：
    # T 成交行 -> A 剩余委托行
    # 所以只有 TradeApplSeqNum < A行 ApplSeqNum 的成交，才是需要加回的首次撮合部分。
    pre_add_summary = (
        active_cj
        .join(
            wt.select([
                "ChannelNo",
                "ApplSeqNum",
                "SecurityID",
                "Side",
            ]),
            on=["ChannelNo", "ApplSeqNum", "SecurityID", "Side"],
            how="inner",
        )
        .filter(pl.col("TradeApplSeqNum") < pl.col("ApplSeqNum"))
        .group_by(["ChannelNo", "ApplSeqNum", "SecurityID", "Side"])
        .agg([
            pl.sum("OrderQty").alias("PreDealQty"),
            pl.last("Price"),
            pl.last("TransactTime"),
        ])
    )

    partial = (
        wt
        .join(
            pre_add_summary.select([
                "ChannelNo",
                "ApplSeqNum",
                "SecurityID",
                "Side",
                "PreDealQty",
            ]),
            on=["ChannelNo", "ApplSeqNum", "SecurityID", "Side"],
            how="inner",
        )
        .with_columns([
            (pl.col("OrderQty") + pl.col("PreDealQty")).alias("OrderQty"),
            pl.lit("主动部分成交").alias("OrderStatus"),
        ])
        .drop("PreDealQty")
    )

    # 3. A 行存在、且 A 行之前没有主动成交：直接保留 A 行 Qty。
    untouched = (
        wt
        .join(
            pre_add_summary.select([
                "ChannelNo",
                "ApplSeqNum",
                "SecurityID",
                "Side",
            ]),
            on=["ChannelNo", "ApplSeqNum", "SecurityID", "Side"],
            how="anti",
        )
        .with_columns(pl.lit("普通委托").alias("OrderStatus"))
    )
    untouched = untouched.select(partial.columns)

    # 4. 没有 A 行的主动完全成交订单：用主动成交汇总构造。
    # 后面计算盘口时再用完整成交扣掉，最终 RemainQty 应为 0。
    active_summary = (
        active_cj
        .group_by(["ChannelNo", "ApplSeqNum", "SecurityID", "Side"])
        .agg([
            pl.sum("OrderQty"),
            pl.last("Price"),
            pl.last("TransactTime"),
        ])
    )

    new = (
        active_summary
        .join(
            wt.select(["ChannelNo", "ApplSeqNum", "SecurityID", "Side"]),
            on=["ChannelNo", "ApplSeqNum", "SecurityID", "Side"],
            how="anti",
        )
        .with_columns(pl.lit("主动完全成交").alias("OrderStatus"))
    )
    new = new.select(partial.columns)

    init_order = pl.concat([partial, untouched, new])
    return init_order

In [51]:
shwt = RestoreOrder(shwt_added, shcj)
shwt

ApplSeqNum,ChannelNo,SecurityID,TransactTime,Price,OrderQty,Side,OrderStatus
i64,i64,i64,time,f64,i64,i32,str
473433,1,600448,09:30:00,3.4,16300,1,"""主动部分成交"""
473586,1,601929,09:30:00,3.71,1800,-1,"""主动部分成交"""
473589,1,603917,09:30:00,14.16,3900,1,"""主动部分成交"""
473672,1,600036,09:30:00,40.71,211100,1,"""主动部分成交"""
473873,1,518680,09:30:00,8.8,50000,-1,"""主动部分成交"""
…,…,…,…,…,…,…,…
6461866,4,600410,10:16:08.530,19.34,300,-1,"""主动完全成交"""
4554429,4,688249,09:57:07.370,33.98,200,-1,"""主动完全成交"""
4460906,1,601717,09:40:48.910,24.96,200,-1,"""主动完全成交"""


In [52]:
shwt.drop('OrderStatus').write_parquet(outpath/'shwt.pq',compression="gzip")

#### 检验还原委托的正确性

In [42]:
def get_shot(exchange):
    shot = pl.read_parquet(Path(L2DATA_PATH)/"raw"/f"{date}"/f"{exchange}shot.pq")
    cols = ['SecurityID','UpdateTime'] + ','.join([f'BidPrice{i},BidVolume{i},AskPrice{i},AskVolume{i}' for i in range(1, 11)]).split(',')
    shot = shot.select(cols)
    closing_shot = shot.sort(['SecurityID', 'UpdateTime']).group_by('SecurityID').last()
    closing_orderbook = pl.concat([
        closing_shot.select(
            pl.col("SecurityID"),
            pl.lit(i).alias("Level"),

            pl.col(f"BidPrice{i}")
            .cast(pl.Float64, strict=False)
            .alias("BidPrice"),

            pl.col(f"BidVolume{i}")
            .cast(pl.Float64, strict=False).cast(pl.Int64, strict=False)
            .alias("BidQty"),

            pl.col(f"AskPrice{i}")
            .cast(pl.Float64, strict=False)
            .alias("AskPrice"),

            pl.col(f"AskVolume{i}")
            .cast(pl.Float64, strict=False).cast(pl.Int64, strict=False)
            .alias("AskQty"),
        )
        for i in range(1, 11)
    ]).sort(["SecurityID", "Level"])
    return closing_orderbook

In [43]:
def get_close_orderbook(
    shwt: pl.DataFrame,
    shcj: pl.DataFrame,
    shcd: pl.DataFrame,
    topn: int = 10,
):
    # 1. 委托订单表
    orders = (
        shwt
        .select([
            "ChannelNo",
            "SecurityID",
            "ApplSeqNum",
            "Price",
            "OrderQty",
            "Side",
        ])
    )

    cj = pl.concat([shcj, shcd])

    # 2. 买方订单成交扣减
    buy_filled = (
        cj
        .select([
            "ChannelNo",
            "SecurityID",
            pl.col("BidApplSeqNum").alias("ApplSeqNum"),
            pl.col("OrderQty").alias("DealQty"),
        ])
        .with_columns(pl.lit(1).alias('Side'))
    )

    # 3. 卖方订单成交扣减
    sell_filled = (
        cj
        .select([
            "ChannelNo",
            "SecurityID",
            pl.col("OfferApplSeqNum").alias("ApplSeqNum"),
            pl.col("OrderQty").alias("DealQty"),
        ])
        .with_columns(pl.lit(-1).alias('Side'))
    )

    # 4. 每张订单的累计成交数量
    filled = (
        pl.concat([buy_filled, sell_filled])
        .group_by(["ChannelNo", "SecurityID", "Side", "ApplSeqNum"])
        .agg(
            pl.col("DealQty").sum().alias("FilledQty")
        )
    )

    # 8. 计算每张订单剩余数量
    alive_orders = (
        orders
        .join(
            filled,
            on=["ChannelNo", "SecurityID", "Side", "ApplSeqNum"],
            how="left",
        )
        .with_columns([
            (
                pl.col("OrderQty")
                - pl.col("FilledQty").fill_null(0)
            ).alias("RemainQty")
        ])
        .filter(pl.col("RemainQty") > 0)
    )

    # 9. 聚合成价格档位
    price_level = (
        alive_orders
        .group_by(["SecurityID", "Side", "Price"])
        .agg(
            pl.col("RemainQty").sum().alias("Qty")
        )
    )

    # 10. 买盘：价格从高到低
    bid_book = (
        price_level
        .filter(pl.col("Side") == 1)
        .with_columns(
            pl.col("Price")
            .rank(method="dense", descending=True)
            .over("SecurityID")
            .cast(pl.Int32)
            .alias("Level")
        )
        .filter(pl.col("Level") <= topn)
        .sort(["SecurityID", "Level"])
        .rename({
            "Price": "BidPrice",
            "Qty": "BidQty",
        })
        .select(["SecurityID", "Level", "BidPrice", "BidQty"])
    )

    # 11. 卖盘：价格从低到高
    ask_book = (
        price_level
        .filter(pl.col("Side") == -1)
        .with_columns(
            pl.col("Price")
            .rank(method="dense", descending=False)
            .over("SecurityID")
            .cast(pl.Int32)
            .alias("Level")
        )
        .filter(pl.col("Level") <= topn)
        .sort(["SecurityID", "Level"])
        .rename({
            "Price": "AskPrice",
            "Qty": "AskQty",
        })
        .select(["SecurityID", "Level", "AskPrice", "AskQty"])
    )

    # 12. 合并买卖盘
    close_book = (
        bid_book
        .join(
            ask_book,
            on=["SecurityID", "Level"],
            how="full",
            coalesce=True,
        )
        .sort(["SecurityID", "Level"])
    )

    return close_book, alive_orders

##### 检查深交所

In [110]:
sz_closing_orderbook = get_shot('sz')

In [ ]:
sz_closing_orderbook.filter(pl.col('SecurityID')==300575)

In [112]:
szcj = pl.read_parquet(outpath/'szcj.pq')
szwt = pl.read_parquet(outpath/'szwt.pq')
szcancel = pl.read_parquet(outpath/'szcancel.pq')

In [113]:
sz_close_book, sz_alive_orders = get_close_orderbook(szwt, szcj, szcancel, topn=10)

In [ ]:
sz_close_book.filter(pl.col('SecurityID')==300575)

##### 检查上交所

In [44]:
sh_closing_orderbook = get_shot('sh')

In [45]:
sh_closing_orderbook.filter(pl.col('SecurityID')==600000)

SecurityID,Level,BidPrice,BidQty,AskPrice,AskQty
i64,i32,f64,i64,f64,i64
600000,1,11.9,254017,11.91,155700
600000,2,11.89,12700,11.92,181200
600000,3,11.88,11400,11.93,39300
600000,4,11.87,21500,11.94,69200
600000,5,11.86,22200,11.95,120871
600000,6,11.85,719800,11.96,166005
600000,7,11.84,133100,11.97,1256300
600000,8,11.83,233700,11.98,202000
600000,9,11.82,46200,11.99,20235400


In [46]:
shwt = pl.read_parquet(outpath/'shwt.pq')
shcj = pl.read_parquet(outpath/'shcj.pq')
shcancel = pl.read_parquet(outpath/'shcancel.pq')

In [53]:
sh_close_book, sh_alive_orders = get_close_orderbook(shwt, shcj, shcancel, topn=40)

In [54]:
sh_close_book.filter(pl.col('SecurityID')==600000)

SecurityID,Level,BidPrice,BidQty,AskPrice,AskQty
i64,i32,f64,i64,f64,i64
600000,1,11.9,254017,11.91,155700
600000,2,11.89,12700,11.92,181200
600000,3,11.88,11400,11.93,39300
600000,4,11.87,21500,11.94,69200
600000,5,11.86,22200,11.95,120871
…,…,…,…,…,…
600000,36,11.55,5600,12.26,136200
600000,37,11.54,4800,12.27,67100
600000,38,11.53,14800,12.28,59700
